# Turkish Public Procurement Intelligence Project

Raw-to-dashboard analysis using Python, DuckDB SQL, Parquet, and Power BI.

- Source rows: 2,370,736
- Source columns: 47
- Stated raw-data coverage: 2010–2024

## How to rerun this notebook

1. Select the repository `.venv` kernel.
2. Restart the kernel.
3. Run every cell from top to bottom without skipping cells.
4. Expect the buyer–supplier history materialization to take about two minutes.
5. Do not interrupt the kernel while a cell shows `[*]`.
6. Review every validation table before exporting.

The raw CSV is never modified. Generated Parquet files are written only to the Git-ignored `data/processed` folder.

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

In [ ]:
ROOT = Path.cwd()

if not (ROOT / ".env").exists() and (ROOT.parent / ".env").exists():
    ROOT = ROOT.parent

if not (ROOT / ".env").exists():
    raise FileNotFoundError(
        "Open the repository root in VS Code before running this file."
    )

load_dotenv(ROOT / ".env")

RAW_CSV = Path(os.environ["PROCUREMENT_RAW_CSV"])
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw file:", RAW_CSV)
print("Raw file exists:", RAW_CSV.exists())
print("Processed folder:", PROCESSED_DIR)

In [ ]:
DUCKDB_TEMP_DIR = ROOT / "data" / "interim" / "duckdb_temp"
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()

duckdb_temp_sql = DUCKDB_TEMP_DIR.as_posix().replace("'", "''")
con.execute(f"SET temp_directory = '{duckdb_temp_sql}'")

print("DuckDB connection opened")
print("DuckDB temporary folder:", DUCKDB_TEMP_DIR)


In [ ]:
raw_csv_sql = RAW_CSV.as_posix().replace("'", "''")

con.execute(
    f"""
    CREATE OR REPLACE VIEW raw_contracts AS
    SELECT *
    FROM read_csv_auto(
        '{raw_csv_sql}',
        all_varchar = true
    )
    """
)

print("Created DuckDB view: raw_contracts")

In [ ]:
row_count = con.execute(
    " SELECT COUNT(*) FROM raw_contracts "
).fetchone()[0]

print(f"Raw rows: {row_count:,}")

In [ ]:
sample = con.execute(
    """
    SELECT
        contract_id,
        tender_name,
        authority,
        province,
        supplier,
        contract_price
    FROM raw_contracts
    LIMIT 10
    """
).df()

sample

In [ ]:
# inspect the schema
schema = con.execute(
    "describe raw_contracts"
).df()
schema

In [ ]:
# List column names
column_names = schema["column_name"].to_list()

print(f"column count: {len(column_names)}")
for name in column_names:
    print(name)

In [ ]:
# View a small selection
con.execute(
    """
    SELECT
        contract_id,
        ikn,
        tender_name,
        authority,
        province,
        okas_code,
        method,
        type,
        contract_date,
        contract_price,
        supplier,
        num_valid_offers
    FROM raw_contracts
    LIMIT 10
    """
).df()

In [ ]:
# %%
missing_critical = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(contract_id) AS missing_contract_id,
        COUNT(*) - COUNT(authority_id) AS missing_authority_id,
        COUNT(*) - COUNT(supplier) AS missing_supplier,
        COUNT(*) - COUNT(contract_date) AS missing_contract_date,
        COUNT(*) - COUNT(contract_price) AS missing_contract_price,
        COUNT(*) - COUNT(okas_code) AS missing_okas_code,
        COUNT(*) - COUNT(num_valid_offers) AS missing_valid_offers
    FROM raw_contracts
    """
).df()

missing_critical

In [ ]:
# Important: empty strings may not be SQL NULL. Check both:
# %%
blank_critical = con.execute(
    """
    SELECT
        SUM(CASE WHEN TRIM(COALESCE(contract_id, '')) = '' THEN 1 ELSE 0 END)
            AS blank_contract_id,
        SUM(CASE WHEN TRIM(COALESCE(supplier, '')) = '' THEN 1 ELSE 0 END)
            AS blank_supplier,
        SUM(CASE WHEN TRIM(COALESCE(contract_price, '')) = '' THEN 1 ELSE 0 END)
            AS blank_contract_price
    FROM raw_contracts
    """
).df()

blank_critical

In [ ]:
# Distinct counts
# %%
distinct_counts = con.execute(
    """
    SELECT
        COUNT(DISTINCT tender_id) AS tenders,
        COUNT(DISTINCT contract_id) AS contracts,
        COUNT(DISTINCT authority_id) AS authorities,
        COUNT(DISTINCT supplier) AS supplier_names,
        COUNT(DISTINCT province) AS provinces,
        COUNT(DISTINCT okas_code) AS okas_codes,
        COUNT(DISTINCT method_code) AS method_codes,
        COUNT(DISTINCT type) AS procurement_types
    FROM raw_contracts
    """
).df()

distinct_counts

In [ ]:
# Category values
# %%
con.execute(
    """
    SELECT
        type,
        COUNT(*) AS rows
    FROM raw_contracts
    GROUP BY type
    ORDER BY rows DESC
    """
).df()

In [ ]:
# method values
con.execute(
    """
    SELECT
        method,
        COUNT(*) AS rows
    FROM raw_contracts
    GROUP BY method
    ORDER BY rows DESC
    """
).df()

In [ ]:
# method values
con.execute(
    """
    SELECT
        method_code,
        COUNT(*) AS rows
    FROM raw_contracts
    GROUP BY method_code
    ORDER BY rows DESC
    """
).df()

In [ ]:
# scope
con.execute(
    """
    SELECT
        scope,
        COUNT(*) AS rows
    FROM raw_contracts
    GROUP BY scope
    ORDER BY rows DESC
    """
).df()

In [ ]:
# status
con.execute(
    """
    select
        status,
        count(*) as contracts
    from raw_contracts
    group by status
    order by contracts desc
    """
).df()

In [ ]:
# province
con.execute(
    """
    select
        province,
        count(*) as contracts
    from raw_contracts
    group by province
    order by contracts desc
    """
).df()

In [ ]:
con.execute(
    """
    select
        is_multi_lot,
        count(*) as contracts
    from raw_contracts
    group by is_multi_lot
    order by contracts desc
    """
).df()

# Determine the row grain

In [ ]:
# contract_id duplication
con.execute(
    """
    select
        contract_id,
        count(*) as rows_per_contract
    from raw_contracts
    group by contract_id
    having count(*) > 1
    order by rows_per_contract desc
    limit 20
    """
).df()

In [ ]:
# compare identifiers
# %%
con.execute(
    """
    SELECT
        tender_id,
        detail_tender_id,
        contract_id,
        announcement_id,
        ikn,
        supplier,
        total_lots_in_tender,
        is_multi_lot
    FROM raw_contracts
    WHERE contract_id IN (
        SELECT contract_id
        FROM raw_contracts
        GROUP BY contract_id
        HAVING COUNT(*) > 1
    )
    ORDER BY contract_id
    LIMIT 100
    """
).df()

In [ ]:
# Test whether the file contains completely identical rows.

exact_duplicate_check = con.execute(
    """
    SELECT
        COUNT(*) AS total_rows,
        (
            SELECT COUNT(*)
            FROM (
                SELECT DISTINCT *
                FROM raw_contracts
            )
        ) AS distinct_full_rows
    FROM raw_contracts
    """
).df()

exact_duplicate_check["exact_duplicate_excess_rows"] = (
    exact_duplicate_check["total_rows"]
    - exact_duplicate_check["distinct_full_rows"]
)

exact_duplicate_check

In [ ]:
# Test the proposed compound key.
# A zero-row result means the combination is unique in the source.

compound_key_duplicates = con.execute(
    """
    SELECT
        tender_id,
        detail_tender_id,
        contract_id,
        supplier,
        COUNT(*) AS duplicate_rows
    FROM raw_contracts
    GROUP BY
        tender_id,
        detail_tender_id,
        contract_id,
        supplier
    HAVING COUNT(*) > 1
    ORDER BY duplicate_rows DESC
    LIMIT 100
    """
).df()

print(
    "Duplicate compound-key groups shown:",
    len(compound_key_duplicates),
)

compound_key_duplicates

# Row-grain conclusion

## Verified conclusion

One raw row represents one observed procurement-award record associated with a tender detail/lot, contract, and supplier. It does not represent one unique tender.

## Evidence

- The source has **2,370,736 rows** but **2,370,077 distinct contract IDs**, so `contract_id` alone is not unique.
- A tender identified by `ikn` can contain multiple lots, contracts, or suppliers.
- The complete-row test finds **2,370,736 distinct full rows**, meaning there are **no exact duplicate source rows**.
- The proposed compound key  
  `tender_id + detail_tender_id + contract_id + supplier`  
  produces **zero duplicate groups**.

## Modeling decision

All source rows are retained. No automatic deduplication is applied. The Power BI fact table receives a generated `contract_row_key` for model relationships, while the verified compound key remains the business-grain evidence. `contract_id` must not be treated as a unique primary key.

# Phase C — Explicit data-type audit

The raw CSV is initially read with every field as text. This phase tests whether numeric, date, and Boolean fields can be converted safely.

Missing or blank values are counted separately from invalid values. An invalid value is a nonblank value that cannot be converted to its proposed type.

Identifiers and classification codes remain strings to preserve their exact representation and possible leading zeros.

In [ ]:
raw_schema = con.execute(
    """
    DESCRIBE raw_contracts
    """
).df()

raw_schema

In [ ]:
# Audit integer conversions
integer_conversion_check = con.execute(
    """
    SELECT
        'okas_count' AS field_name,
        'INTEGER' AS proposed_type,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(okas_count), '') IS NULL
        ) AS missing_or_blank,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(okas_count), '') IS NOT NULL
              AND TRY_CAST(TRIM(okas_count) AS INTEGER) IS NULL
        ) AS invalid_values
    FROM raw_contracts

    UNION ALL

    SELECT
        'document_count',
        'INTEGER',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(document_count), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(document_count), '') IS NOT NULL
              AND TRY_CAST(TRIM(document_count) AS INTEGER) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'days_announce_to_contract',
        'INTEGER',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(days_announce_to_contract), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(days_announce_to_contract), '') IS NOT NULL
              AND TRY_CAST(TRIM(days_announce_to_contract) AS INTEGER) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'num_offers',
        'INTEGER',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(num_offers), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(num_offers), '') IS NOT NULL
              AND TRY_CAST(TRIM(num_offers) AS INTEGER) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'num_valid_offers',
        'INTEGER',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(num_valid_offers), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(num_valid_offers), '') IS NOT NULL
              AND TRY_CAST(TRIM(num_valid_offers) AS INTEGER) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'total_lots_in_tender',
        'INTEGER',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(total_lots_in_tender), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(total_lots_in_tender), '') IS NOT NULL
              AND TRY_CAST(TRIM(total_lots_in_tender) AS INTEGER) IS NULL
        )
    FROM raw_contracts
    """
).df()

integer_conversion_check

In [ ]:
# audit decimal conversion
decimal_conversion_check = con.execute(
    """
    SELECT
        'total_estimated_cost' AS field_name,
        'DOUBLE' AS proposed_type,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(total_estimated_cost), '') IS NULL
        ) AS missing_or_blank,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(total_estimated_cost), '') IS NOT NULL
              AND TRY_CAST(TRIM(total_estimated_cost) AS DOUBLE) IS NULL
        ) AS invalid_values
    FROM raw_contracts

    UNION ALL

    SELECT
        'lot_estimated_cost',
        'DOUBLE',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(lot_estimated_cost), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(lot_estimated_cost), '') IS NOT NULL
              AND TRY_CAST(TRIM(lot_estimated_cost) AS DOUBLE) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'contract_price',
        'DOUBLE',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(contract_price), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(contract_price), '') IS NOT NULL
              AND TRY_CAST(TRIM(contract_price) AS DOUBLE) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'rebate',
        'DOUBLE',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(rebate), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(rebate), '') IS NOT NULL
              AND TRY_CAST(TRIM(rebate) AS DOUBLE) IS NULL
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'log_ratio',
        'DOUBLE',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(log_ratio), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(log_ratio), '') IS NOT NULL
              AND TRY_CAST(TRIM(log_ratio) AS DOUBLE) IS NULL
        )
    FROM raw_contracts
    """
).df()

decimal_conversion_check

In [ ]:
# Inspect Boolean raw values
boolean_values = con.execute(
    """
    SELECT
        'is_electronic' AS field_name,
        is_electronic AS raw_value,
        COUNT(*) AS rows
    FROM raw_contracts
    GROUP BY is_electronic

    UNION ALL

    SELECT
        'is_partial',
        is_partial,
        COUNT(*)
    FROM raw_contracts
    GROUP BY is_partial

    UNION ALL

    SELECT
        'is_invitation_only',
        is_invitation_only,
        COUNT(*)
    FROM raw_contracts
    GROUP BY is_invitation_only

    UNION ALL

    SELECT
        'lot_estimate_missing',
        lot_estimate_missing,
        COUNT(*)
    FROM raw_contracts
    GROUP BY lot_estimate_missing

    UNION ALL

    SELECT
        'is_multi_lot',
        is_multi_lot,
        COUNT(*)
    FROM raw_contracts
    GROUP BY is_multi_lot

    ORDER BY field_name, raw_value
    """
).df()

boolean_values

## Phase A findings

- The raw source contains **2,370,736 rows and 47 columns**.
- All raw fields are deliberately registered as `VARCHAR` so type conversion can be audited explicitly.
- There are **1,557,241 distinct tender IDs** and **2,370,077 distinct contract IDs**. Contract ID therefore does not uniquely identify every raw row.
- The data contain **29,259 authority IDs**, **230,369 observed supplier-name strings**, **82 province-category values**, and **6,929 observed OKAS codes**.
- Critical missingness includes **12,911 authority IDs**, **75,139 supplier values**, **911,382 contract dates**, and **399,818 OKAS codes**. Contract ID, contract price, and valid-offer count are present in all rows.
- The procurement-type distribution is dominated by `Mal`, followed by `Hizmet` and `Yapım`; 12,911 rows have no procurement type.
- Open procedure (`İhale Usulü: Açık`) is the largest procedure category.
- **999,542 rows** are marked as multi-lot and **1,371,194 rows** are not.
- The 82 observed province-category values require a later geographic review because Turkey has 81 provinces; the additional category may represent a nonstandard or external grouping.

These are source-row counts. They should not automatically be described as unique contracts.

In [ ]:
# Count invalid Boolean values
boolean_conversion_check = con.execute(
    """
    SELECT
        'is_electronic' AS field_name,
        'BOOLEAN' AS proposed_type,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_electronic), '') IS NULL
        ) AS missing_or_blank,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_electronic), '') IS NOT NULL
              AND TRIM(is_electronic) NOT IN ('0', '1')
        ) AS invalid_values
    FROM raw_contracts

    UNION ALL

    SELECT
        'is_partial',
        'BOOLEAN',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_partial), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_partial), '') IS NOT NULL
              AND TRIM(is_partial) NOT IN ('0', '1')
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'is_invitation_only',
        'BOOLEAN',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_invitation_only), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_invitation_only), '') IS NOT NULL
              AND TRIM(is_invitation_only) NOT IN ('0', '1')
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'lot_estimate_missing',
        'BOOLEAN',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(lot_estimate_missing), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(lot_estimate_missing), '') IS NOT NULL
              AND TRIM(lot_estimate_missing) NOT IN ('0', '1')
        )
    FROM raw_contracts

    UNION ALL

    SELECT
        'is_multi_lot',
        'BOOLEAN',
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_multi_lot), '') IS NULL
        ),
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(is_multi_lot), '') IS NOT NULL
              AND TRIM(is_multi_lot) NOT IN ('0', '1')
        )
    FROM raw_contracts
    """
).df()

boolean_conversion_check

In [ ]:
# view date examples
date_examples = con.execute(
    """
    SELECT
        tender_datetime,
        tender_date,
        tender_announcement_date,
        result_announcement_date,
        contract_date
    FROM raw_contracts
    WHERE tender_datetime IS NOT NULL
       OR tender_date IS NOT NULL
       OR tender_announcement_date IS NOT NULL
       OR result_announcement_date IS NOT NULL
       OR contract_date IS NOT NULL
    LIMIT 20
    """
).df()

date_examples

In [ ]:
# Helper SQL expressions used to parse the date columns safely.
# COALESCE returns the first parsing attempt that succeeds.

tender_datetime_parse = """
COALESCE(
    TRY_CAST(NULLIF(TRIM(tender_datetime), '') AS TIMESTAMP),
    TRY_STRPTIME(NULLIF(TRIM(tender_datetime), ''), '%d.%m.%Y %H:%M')
)
"""

date_parse_expressions = {
    "tender_date": """
        COALESCE(
            TRY_CAST(NULLIF(TRIM(tender_date), '') AS DATE),
            CAST(TRY_STRPTIME(NULLIF(TRIM(tender_date), ''), '%m/%d/%Y') AS DATE)
        )
    """,
    "tender_announcement_date": """
        COALESCE(
            TRY_CAST(NULLIF(TRIM(tender_announcement_date), '') AS DATE),
            CAST(TRY_STRPTIME(NULLIF(TRIM(tender_announcement_date), ''), '%m/%d/%Y') AS DATE)
        )
    """,
    "result_announcement_date": """
        COALESCE(
            TRY_CAST(NULLIF(TRIM(result_announcement_date), '') AS DATE),
            CAST(TRY_STRPTIME(NULLIF(TRIM(result_announcement_date), ''), '%m/%d/%Y') AS DATE)
        )
    """,
    "contract_date": """
        COALESCE(
            TRY_CAST(NULLIF(TRIM(contract_date), '') AS DATE),
            CAST(TRY_STRPTIME(NULLIF(TRIM(contract_date), ''), '%m/%d/%Y') AS DATE)
        )
    """,
}

print("Created reusable date-parsing expressions.")

In [ ]:
# Audit the corrected date conversions.
# A value is invalid only when it is nonblank and every safe parsing attempt fails.

date_audit_parts = [
    f"""
    SELECT
        'tender_datetime' AS field_name,
        'TIMESTAMP' AS proposed_type,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(tender_datetime), '') IS NULL
        ) AS missing_or_blank,
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(tender_datetime), '') IS NOT NULL
              AND ({tender_datetime_parse}) IS NULL
        ) AS invalid_values
    FROM raw_contracts
    """
]

for field_name, parse_expression in date_parse_expressions.items():
    date_audit_parts.append(
        f"""
        SELECT
            '{field_name}' AS field_name,
            'DATE' AS proposed_type,
            COUNT(*) FILTER (
                WHERE NULLIF(TRIM({field_name}), '') IS NULL
            ) AS missing_or_blank,
            COUNT(*) FILTER (
                WHERE NULLIF(TRIM({field_name}), '') IS NOT NULL
                  AND ({parse_expression}) IS NULL
            ) AS invalid_values
        FROM raw_contracts
        """
    )

date_conversion_check = con.execute(
    "\nUNION ALL\n".join(date_audit_parts)
).df()

date_conversion_check

In [ ]:
# Display any nonblank date values that still cannot be parsed.
# An empty result means the proposed parsing rules cover all observed values.

invalid_date_examples = con.execute(
    f"""
    SELECT
        'tender_datetime' AS field_name,
        tender_datetime AS raw_value,
        COUNT(*) AS rows
    FROM raw_contracts
    WHERE NULLIF(TRIM(tender_datetime), '') IS NOT NULL
      AND ({tender_datetime_parse}) IS NULL
    GROUP BY tender_datetime

    UNION ALL

    SELECT
        'tender_date',
        tender_date,
        COUNT(*)
    FROM raw_contracts
    WHERE NULLIF(TRIM(tender_date), '') IS NOT NULL
      AND ({date_parse_expressions['tender_date']}) IS NULL
    GROUP BY tender_date

    UNION ALL

    SELECT
        'tender_announcement_date',
        tender_announcement_date,
        COUNT(*)
    FROM raw_contracts
    WHERE NULLIF(TRIM(tender_announcement_date), '') IS NOT NULL
      AND ({date_parse_expressions['tender_announcement_date']}) IS NULL
    GROUP BY tender_announcement_date

    UNION ALL

    SELECT
        'result_announcement_date',
        result_announcement_date,
        COUNT(*)
    FROM raw_contracts
    WHERE NULLIF(TRIM(result_announcement_date), '') IS NOT NULL
      AND ({date_parse_expressions['result_announcement_date']}) IS NULL
    GROUP BY result_announcement_date

    UNION ALL

    SELECT
        'contract_date',
        contract_date,
        COUNT(*)
    FROM raw_contracts
    WHERE NULLIF(TRIM(contract_date), '') IS NOT NULL
      AND ({date_parse_expressions['contract_date']}) IS NULL
    GROUP BY contract_date

    ORDER BY rows DESC
    LIMIT 50
    """
).df()

invalid_date_examples

In [ ]:
# Final Phase C summary.

print("INTEGER CONVERSIONS")
display(integer_conversion_check)

print("DECIMAL CONVERSIONS")
display(decimal_conversion_check)

print("BOOLEAN CONVERSIONS")
display(boolean_conversion_check)

print("DATE CONVERSIONS")
display(date_conversion_check)

## Phase C completion rule

Phase C is complete when:

- identifiers and classification codes remain text;
- every `invalid_values` count is zero, or each exception is documented;
- Boolean fields contain only `0`, `1`, or missing values;
- the date audit uses the actual source formats;
- `invalid_date_examples` is empty, or its remaining values are documented.

Missing values are not automatically errors. They describe data availability and will be measured later.

## Phase C findings

- All six proposed integer fields convert successfully with **zero invalid nonblank values**.
- All five proposed decimal fields convert successfully with **zero invalid nonblank values**.
- All five binary fields contain only `0` and `1`, with **zero invalid values**.
- `tender_datetime` parses successfully as a timestamp.
- The four date columns are primarily ISO-formatted (`YYYY-MM-DD`), with a fallback for slash-formatted dates.
- Every nonblank date value parses successfully; **zero nonblank dates are lost**.
- Identifiers and classification codes remain text to preserve exact values and possible leading zeros.

Phase C is complete. Missing values remain documented data-availability issues rather than conversion errors.

# Phase D — Create the cleaned contract view

This phase creates `clean_contracts` without changing `raw_contracts`.

The cleaned view:

- trims text and changes blank strings to SQL `NULL`;
- preserves identifiers and codes as text;
- safely converts counts, prices, dates, and flags;
- retains raw date values beside parsed dates for auditing;
- does not filter or delete any rows.

A DuckDB view is a saved query, not a second physical copy of the 1.8 GB CSV.

In [ ]:
# Create the cleaned view.
# TRY_CAST returns NULL instead of stopping the notebook when conversion fails.

con.execute(
    """
    CREATE OR REPLACE VIEW clean_contracts AS
    SELECT
        -- Identifiers: retain as text
        NULLIF(TRIM(tender_id), '') AS tender_id,
        NULLIF(TRIM(detail_tender_id), '') AS detail_tender_id,
        NULLIF(TRIM(contract_id), '') AS contract_id,
        NULLIF(TRIM(announcement_id), '') AS announcement_id,
        NULLIF(TRIM(ikn), '') AS ikn,

        -- Buyer, geography, and descriptive text
        NULLIF(TRIM(tender_name), '') AS tender_name,
        NULLIF(TRIM(authority), '') AS authority,
        NULLIF(TRIM(authority_id), '') AS authority_id,
        NULLIF(TRIM(province), '') AS province,
        NULLIF(TRIM(authority_district), '') AS authority_district,
        NULLIF(TRIM(parent_authority), '') AS parent_authority,
        NULLIF(TRIM(top_authority_code), '') AS top_authority_code,
        NULLIF(TRIM(top_authority_name), '') AS top_authority_name,

        -- Product/classification fields
        NULLIF(TRIM(okas_code), '') AS okas_code,
        NULLIF(TRIM(okas_desc), '') AS okas_desc,
        NULLIF(TRIM(okas_code_search), '') AS okas_code_search,
        NULLIF(TRIM(okas_codes_all), '') AS okas_codes_all,
        NULLIF(TRIM(okas_names_all), '') AS okas_names_all,
        TRY_CAST(NULLIF(TRIM(okas_count), '') AS INTEGER) AS okas_count,

        -- Procedure and status fields
        NULLIF(TRIM(method), '') AS method,
        NULLIF(TRIM(method_code), '') AS method_code,
        NULLIF(TRIM(type), '') AS procurement_type,
        NULLIF(TRIM(scope), '') AS scope,
        NULLIF(TRIM(characteristics), '') AS characteristics,
        TRY_CAST(NULLIF(TRIM(document_count), '') AS INTEGER) AS document_count,
        NULLIF(TRIM(status), '') AS status,
        NULLIF(TRIM(status_code), '') AS status_code,

        -- Flags: convert source 0/1 values to true/false
        CASE TRIM(is_electronic)
            WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL
        END AS is_electronic,
        CASE TRIM(is_partial)
            WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL
        END AS is_partial,
        CASE TRIM(is_invitation_only)
            WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL
        END AS is_invitation_only,
        CASE TRIM(lot_estimate_missing)
            WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL
        END AS lot_estimate_missing,
        CASE TRIM(is_multi_lot)
            WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL
        END AS is_multi_lot,

        -- Preserve source date text for traceability
        NULLIF(TRIM(tender_datetime), '') AS tender_datetime_raw,
        NULLIF(TRIM(tender_date), '') AS tender_date_raw,
        NULLIF(TRIM(tender_announcement_date), '') AS tender_announcement_date_raw,
        NULLIF(TRIM(result_announcement_date), '') AS result_announcement_date_raw,
        NULLIF(TRIM(contract_date), '') AS contract_date_raw,

        -- Parsed dates
        COALESCE(
            TRY_CAST(NULLIF(TRIM(tender_datetime), '') AS TIMESTAMP),
            TRY_STRPTIME(
                NULLIF(TRIM(tender_datetime), ''),
                '%d.%m.%Y %H:%M'
            )
        ) AS tender_datetime,
        COALESCE(
            TRY_CAST(NULLIF(TRIM(tender_date), '') AS DATE),
            CAST(TRY_STRPTIME(
                NULLIF(TRIM(tender_date), ''),
                '%m/%d/%Y'
            ) AS DATE)
        ) AS tender_date,
        COALESCE(
            TRY_CAST(NULLIF(TRIM(tender_announcement_date), '') AS DATE),
            CAST(TRY_STRPTIME(
                NULLIF(TRIM(tender_announcement_date), ''),
                '%m/%d/%Y'
            ) AS DATE)
        ) AS tender_announcement_date,
        COALESCE(
            TRY_CAST(NULLIF(TRIM(result_announcement_date), '') AS DATE),
            CAST(TRY_STRPTIME(
                NULLIF(TRIM(result_announcement_date), ''),
                '%m/%d/%Y'
            ) AS DATE)
        ) AS result_announcement_date,
        COALESCE(
            TRY_CAST(NULLIF(TRIM(contract_date), '') AS DATE),
            CAST(TRY_STRPTIME(
                NULLIF(TRIM(contract_date), ''),
                '%m/%d/%Y'
            ) AS DATE)
        ) AS contract_date,

        -- Counts and financial measures
        TRY_CAST(
            NULLIF(TRIM(days_announce_to_contract), '') AS INTEGER
        ) AS days_announce_to_contract,
        TRY_CAST(
            NULLIF(TRIM(total_estimated_cost), '') AS DECIMAL(20, 2)
        ) AS total_estimated_cost,
        TRY_CAST(
            NULLIF(TRIM(lot_estimated_cost), '') AS DECIMAL(20, 2)
        ) AS lot_estimated_cost,
        TRY_CAST(
            NULLIF(TRIM(contract_price), '') AS DECIMAL(20, 2)
        ) AS contract_price,
        NULLIF(TRIM(supplier), '') AS supplier,
        TRY_CAST(NULLIF(TRIM(num_offers), '') AS INTEGER) AS num_offers,
        TRY_CAST(
            NULLIF(TRIM(num_valid_offers), '') AS INTEGER
        ) AS num_valid_offers,

        -- Source-derived measures retained for later validation
        TRY_CAST(NULLIF(TRIM(rebate), '') AS DOUBLE) AS rebate_raw_derived,
        TRY_CAST(NULLIF(TRIM(log_ratio), '') AS DOUBLE) AS log_ratio_raw_derived,

        TRY_CAST(
            NULLIF(TRIM(total_lots_in_tender), '') AS INTEGER
        ) AS total_lots_in_tender
    FROM raw_contracts
    """
)

print("Created DuckDB view: clean_contracts")

In [ ]:
# Check 1: cleaning a view must not change the number of rows.

raw_count = con.execute(
    "SELECT COUNT(*) FROM raw_contracts"
).fetchone()[0]

clean_count = con.execute(
    "SELECT COUNT(*) FROM clean_contracts"
).fetchone()[0]

print(f"Raw rows:   {raw_count:,}")
print(f"Clean rows: {clean_count:,}")
print("Row counts match:", raw_count == clean_count)

assert raw_count == clean_count, "Unexpected row loss or duplication."

In [ ]:
# Check 2: inspect the cleaned schema and confirm the new data types.

clean_schema = con.execute(
    "DESCRIBE clean_contracts"
).df()

clean_schema

In [ ]:
# Check 3: inspect readable examples from the cleaned view.

clean_sample = con.execute(
    """
    SELECT
        ikn,
        authority,
        province,
        supplier,
        procurement_type,
        contract_date_raw,
        contract_date,
        contract_price,
        num_offers,
        num_valid_offers,
        is_electronic,
        is_multi_lot
    FROM clean_contracts
    LIMIT 20
    """
).df()

clean_sample

In [ ]:
# Check 4: verify that no nonblank source date was lost during parsing.

date_loss_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE tender_datetime_raw IS NOT NULL
              AND tender_datetime IS NULL
        ) AS lost_tender_datetime,
        COUNT(*) FILTER (
            WHERE tender_date_raw IS NOT NULL
              AND tender_date IS NULL
        ) AS lost_tender_date,
        COUNT(*) FILTER (
            WHERE tender_announcement_date_raw IS NOT NULL
              AND tender_announcement_date IS NULL
        ) AS lost_tender_announcement_date,
        COUNT(*) FILTER (
            WHERE result_announcement_date_raw IS NOT NULL
              AND result_announcement_date IS NULL
        ) AS lost_result_announcement_date,
        COUNT(*) FILTER (
            WHERE contract_date_raw IS NOT NULL
              AND contract_date IS NULL
        ) AS lost_contract_date
    FROM clean_contracts
    """
).df()

date_loss_check

In [ ]:
# Check 5: inspect date ranges for impossible or surprising years.

date_range_check = con.execute(
    """
    SELECT
        MIN(tender_date) AS minimum_tender_date,
        MAX(tender_date) AS maximum_tender_date,
        MIN(contract_date) AS minimum_contract_date,
        MAX(contract_date) AS maximum_contract_date
    FROM clean_contracts
    """
).df()

date_range_check

In [ ]:
# Check 6: count suspicious numeric relationships.
# These are investigation flags; this cell does not remove any records.

numeric_quality_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE contract_price < 0
        ) AS negative_contract_prices,
        COUNT(*) FILTER (
            WHERE total_estimated_cost < 0
        ) AS negative_total_estimated_costs,
        COUNT(*) FILTER (
            WHERE lot_estimated_cost < 0
        ) AS negative_lot_estimated_costs,
        COUNT(*) FILTER (
            WHERE num_offers < 0
        ) AS negative_offer_counts,
        COUNT(*) FILTER (
            WHERE num_valid_offers < 0
        ) AS negative_valid_offer_counts,
        COUNT(*) FILTER (
            WHERE num_valid_offers > num_offers
        ) AS valid_offers_exceed_all_offers,
        COUNT(*) FILTER (
            WHERE total_lots_in_tender <= 0
        ) AS nonpositive_total_lots
    FROM clean_contracts
    """
).df()

numeric_quality_check

## Phase D findings

- Raw rows: **2,370,736**.
- Clean rows: **2,370,736**; cleaning preserves every source row.
- Nonblank date values lost during parsing: **0** across all five audited date fields.
- Tender dates range from **1911-12-10 to 2024-12-31**. The 1911 value is an anomalous source date and is flagged for review.
- Contract dates range from **2015-01-02 to 2026-04-01**; **12,639 contract dates** fall after the stated 2024 coverage.
- No negative prices, estimated costs, offer counts, or nonpositive lot totals were found.
- **49 rows** have `num_valid_offers > num_offers`; these remain visible as data-quality exceptions.

`clean_contracts` standardizes representation without filtering or overwriting the raw source.

# Phase E — Analytical variables

This phase derives analysis-ready variables from `clean_contracts`.

Important definitions:

- `analytical_estimated_cost` uses lot cost for multi-lot records when available; otherwise it uses total estimated cost.
- `price_to_estimate_ratio = contract_price / analytical_estimated_cost`.
- `rebate = 1 - price_to_estimate_ratio`.
- missing valid-offer counts remain missing and are never changed to zero.
- `contract_date_clean` is the parsed contract date only.
- `analysis_date` uses contract date first, then result-announcement date, then tender date. This fallback makes time analysis possible while preserving the original date fields.
- supplier standardization only trims, uppercases, and collapses whitespace. It does not merge legally distinct names.
- `okas_code_clean` contains digits only, and `okas2` is derived only when at least two digits exist.

In [ ]:
# Inspect OKAS code lengths before deriving the two-digit group.

okas_length_profile = con.execute(
    """
    SELECT
        LENGTH(
            REGEXP_REPLACE(
                COALESCE(okas_code, ''),
                '[^0-9]',
                '',
                'g'
            )
        ) AS digit_length,
        COUNT(*) AS rows,
        COUNT(DISTINCT okas_code) AS distinct_raw_codes
    FROM clean_contracts
    GROUP BY digit_length
    ORDER BY digit_length
    """
).df()

okas_length_profile

In [ ]:
# Create the analytical view.

con.execute(
    """
    CREATE OR REPLACE VIEW analytical_contracts AS
    WITH standardized AS (
        SELECT
            c.*,

            CASE
                WHEN is_multi_lot
                 AND lot_estimated_cost > 0
                    THEN lot_estimated_cost
                WHEN total_estimated_cost > 0
                    THEN total_estimated_cost
                ELSE NULL
            END AS analytical_estimated_cost,

            UPPER(
                REGEXP_REPLACE(
                    NULLIF(TRIM(supplier), ''),
                    '\\s+',
                    ' ',
                    'g'
                )
            ) AS supplier_clean,

            okas_code AS okas_code_raw,
            NULLIF(
                REGEXP_REPLACE(
                    COALESCE(okas_code, ''),
                    '[^0-9]',
                    '',
                    'g'
                ),
                ''
            ) AS okas_code_clean,

            contract_date AS contract_date_clean,
            COALESCE(
                contract_date,
                result_announcement_date,
                tender_date
            ) AS analysis_date,

            CASE
                WHEN contract_date IS NOT NULL THEN 'contract_date'
                WHEN result_announcement_date IS NOT NULL
                    THEN 'result_announcement_date'
                WHEN tender_date IS NOT NULL THEN 'tender_date'
                ELSE NULL
            END AS analysis_date_source
        FROM clean_contracts AS c
    )
    SELECT
        standardized.*,

        contract_price
            / NULLIF(analytical_estimated_cost, 0)
            AS price_to_estimate_ratio,

        1 - (
            contract_price
            / NULLIF(analytical_estimated_cost, 0)
        ) AS rebate,

        num_valid_offers IS NULL AS valid_offers_missing,
        num_valid_offers = 0 AS zero_valid_offers,
        num_valid_offers = 1 AS single_bid,
        num_valid_offers > 0 AS positive_valid_offers,

        contract_price > analytical_estimated_cost
            AS contract_above_estimate,
        (
            contract_price
            / NULLIF(analytical_estimated_cost, 0)
        ) > 10 AS extreme_price_to_estimate_ratio,
        tender_date < DATE '2010-01-01'
            OR tender_date > DATE '2024-12-31'
            AS tender_date_outside_expected_coverage,
        contract_date_clean < DATE '2010-01-01'
            OR contract_date_clean > DATE '2024-12-31'
            AS contract_date_outside_expected_coverage,
        analysis_date < DATE '2010-01-01'
            OR analysis_date > DATE '2024-12-31'
            AS analysis_date_outside_expected_coverage,

        EXTRACT(YEAR FROM contract_date_clean)::INTEGER
            AS contract_year,
        EXTRACT(MONTH FROM contract_date_clean)::INTEGER
            AS contract_month,
        EXTRACT(YEAR FROM analysis_date)::INTEGER
            AS analysis_year,
        EXTRACT(MONTH FROM analysis_date)::INTEGER
            AS analysis_month,

        CASE
            WHEN LENGTH(okas_code_clean) >= 2
                THEN LEFT(okas_code_clean, 2)
            ELSE NULL
        END AS okas2
    FROM standardized
    """
)

print("Created DuckDB view: analytical_contracts")

In [ ]:
# Validate Phase E row preservation and derived values.

phase_e_validation = con.execute(
    """
    SELECT
        COUNT(*) AS analytical_rows,
        COUNT(*) FILTER (
            WHERE analytical_estimated_cost IS NULL
        ) AS missing_analytical_estimated_cost,
        COUNT(*) FILTER (
            WHERE analytical_estimated_cost <= 0
        ) AS nonpositive_analytical_estimated_cost,
        COUNT(*) FILTER (
            WHERE price_to_estimate_ratio < 0
        ) AS negative_price_ratio,
        COUNT(*) FILTER (
            WHERE contract_above_estimate
        ) AS contract_above_estimate_count,
        COUNT(*) FILTER (
            WHERE extreme_price_to_estimate_ratio
        ) AS extreme_price_ratio_count,
        COUNT(*) FILTER (
            WHERE tender_date_outside_expected_coverage
        ) AS tender_dates_outside_expected_coverage,
        COUNT(*) FILTER (
            WHERE contract_date_outside_expected_coverage
        ) AS contract_dates_outside_expected_coverage,
        COUNT(*) FILTER (
            WHERE valid_offers_missing
        ) AS missing_valid_offer_count,
        COUNT(*) FILTER (
            WHERE zero_valid_offers
        ) AS zero_valid_offer_count,
        COUNT(*) FILTER (
            WHERE single_bid
        ) AS single_bid_count,
        COUNT(*) FILTER (
            WHERE analysis_date IS NULL
        ) AS missing_analysis_date,
        COUNT(*) FILTER (
            WHERE analysis_date < DATE '2010-01-01'
               OR analysis_date > DATE '2024-12-31'
        ) AS analysis_dates_outside_expected_coverage,
        COUNT(*) FILTER (
            WHERE okas2 IS NULL
        ) AS missing_okas2
    FROM analytical_contracts
    """
).df()

phase_e_validation

In [ ]:
# Compare the calculated rebate with the source-derived rebate.
# Median and percentile differences are more informative than the mean
# because a small number of extreme source values can dominate the mean.

derived_measure_check = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE rebate IS NOT NULL
        ) AS calculated_rebate_rows,
        COUNT(*) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
              AND ABS(rebate - rebate_raw_derived) <= 0.000001
        ) AS source_match_within_tolerance,
        COUNT(*) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
              AND ABS(rebate - rebate_raw_derived) > 0.000001
        ) AS source_mismatch,
        COUNT(*) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
              AND ABS(rebate - rebate_raw_derived) > 1
        ) AS source_difference_above_one,
        MEDIAN(
            ABS(rebate - rebate_raw_derived)
        ) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
        ) AS median_absolute_difference,
        QUANTILE_CONT(
            ABS(rebate - rebate_raw_derived),
            0.99
        ) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
        ) AS p99_absolute_difference,
        MAX(
            ABS(rebate - rebate_raw_derived)
        ) FILTER (
            WHERE rebate IS NOT NULL
              AND rebate_raw_derived IS NOT NULL
        ) AS maximum_absolute_difference
    FROM analytical_contracts
    """
).df()

derived_measure_check

## Phase E findings

- All **2,370,736 rows** remain in `analytical_contracts`.
- Analytical estimated cost is unavailable for **291,330 rows** and nonpositive for **0 rows**.
- Valid-offer count is present for every row: **31,052** rows report zero valid offers and **925,594** report exactly one valid offer.
- OKAS2 cannot be derived for **399,818 rows**, matching the missing OKAS-code issue found in Phase A.
- `analysis_date` is available for every row, but **12,749 rows** fall after the expected 2010–2024 coverage. The date dimension retains these observations and the fact table flags them.
- One tender date is before 2010; **12,639 contract dates** are after 2024.
- Rebate can be calculated for **2,079,406 rows**. Of these, **1,871,764** match the source-derived rebate within `0.000001`, while **207,642** differ.
- The median absolute rebate difference is effectively zero, but **696 records** differ by more than 1 and the maximum difference is extreme. The source-provided rebate is therefore retained only for audit, while the independently defined `rebate` is used for analysis.
- **149,146 rows** have contract price above the selected estimate and therefore a negative calculated rebate. These are flagged rather than removed.

These flags must be available on the Power BI data-quality page and as optional report filters.

# Phase F — Buyer–supplier histories

History is ordered by:

1. `analysis_date`;
2. `contract_id`;
3. `tender_id`;
4. `detail_tender_id`.

This gives deterministic ordering. Records on the same date are still ordered by identifiers, so “previous” means previous in the observed data order, not necessarily a legally earlier event during that day.

Supplier names receive conservative standardization only. Consortiums and spelling variants are not automatically merged.

`final_contracts` is materialized as a temporary DuckDB table after the window calculations. This prevents every later validation and export from recalculating millions of history rows. It disappears when the notebook kernel is restarted and can be rebuilt by rerunning the cells.


In [ ]:
# Inspect common supplier names and consortium-like text before building histories.

supplier_name_review = con.execute(
    """
    SELECT
        supplier,
        supplier_clean,
        COUNT(*) AS rows
    FROM analytical_contracts
    WHERE supplier_clean IS NOT NULL
    GROUP BY supplier, supplier_clean
    ORDER BY rows DESC
    LIMIT 30
    """
).df()

supplier_name_review

In [ ]:
# Create supplier history variables with DuckDB window functions.

con.execute(
    """
    CREATE OR REPLACE VIEW contract_histories AS
    SELECT
        a.*,

        CASE
            WHEN supplier_clean IS NULL OR analysis_date IS NULL
                THEN NULL
            ELSE ROW_NUMBER() OVER (
                PARTITION BY supplier_clean
                ORDER BY
                    analysis_date,
                    COALESCE(contract_id, ''),
                    COALESCE(tender_id, ''),
                    COALESCE(detail_tender_id, '')
            ) - 1
        END AS previous_supplier_wins_nationally,

        CASE
            WHEN supplier_clean IS NULL
              OR authority_id IS NULL
              OR analysis_date IS NULL
                THEN NULL
            ELSE ROW_NUMBER() OVER (
                PARTITION BY supplier_clean, authority_id
                ORDER BY
                    analysis_date,
                    COALESCE(contract_id, ''),
                    COALESCE(tender_id, ''),
                    COALESCE(detail_tender_id, '')
            ) - 1
        END AS previous_supplier_buyer_wins,

        CASE
            WHEN supplier_clean IS NULL
              OR okas2 IS NULL
              OR analysis_date IS NULL
                THEN NULL
            ELSE ROW_NUMBER() OVER (
                PARTITION BY supplier_clean, okas2
                ORDER BY
                    analysis_date,
                    COALESCE(contract_id, ''),
                    COALESCE(tender_id, ''),
                    COALESCE(detail_tender_id, '')
            ) - 1
        END AS previous_supplier_okas2_wins,

        CASE
            WHEN supplier_clean IS NULL
              OR authority_id IS NULL
              OR okas2 IS NULL
              OR analysis_date IS NULL
                THEN NULL
            ELSE ROW_NUMBER() OVER (
                PARTITION BY supplier_clean, authority_id, okas2
                ORDER BY
                    analysis_date,
                    COALESCE(contract_id, ''),
                    COALESCE(tender_id, ''),
                    COALESCE(detail_tender_id, '')
            ) - 1
        END AS previous_supplier_buyer_okas2_wins,

        CASE
            WHEN supplier_clean IS NULL THEN NULL
            ELSE MIN(analysis_year) OVER (
                PARTITION BY supplier_clean
            )
        END AS first_observed_supplier_year,

        CASE
            WHEN supplier_clean IS NULL OR analysis_date IS NULL
                THEN NULL
            ELSE MIN(analysis_date) OVER (
                PARTITION BY supplier_clean, authority_id
            )
        END AS first_observed_buyer_supplier_date
    FROM analytical_contracts AS a
    """
)

print("Created DuckDB view: contract_histories")

In [ ]:
# Add readable returning-supplier and relationship-duration variables.

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE final_contracts AS
    SELECT
        h.*,

        CASE
            WHEN previous_supplier_wins_nationally IS NULL THEN NULL
            WHEN previous_supplier_wins_nationally = 0 THEN 'New'
            ELSE 'Returning'
        END AS supplier_status,

        CASE
            WHEN analysis_date IS NULL
              OR first_observed_buyer_supplier_date IS NULL
                THEN NULL
            ELSE DATE_DIFF(
                'day',
                first_observed_buyer_supplier_date,
                analysis_date
            )
        END AS buyer_supplier_relationship_days,

        CASE
            WHEN authority_id IS NOT NULL
                THEN 'AUTH_ID:' || authority_id
            WHEN authority IS NOT NULL
                THEN 'AUTH_NAME:' || MD5(UPPER(authority))
            ELSE 'AUTH:UNKNOWN'
        END AS authority_key,

        CASE
            WHEN supplier_clean IS NOT NULL
                THEN 'SUP:' || MD5(supplier_clean)
            ELSE 'SUP:UNKNOWN'
        END AS supplier_key,

        'GEO:' || MD5(
            COALESCE(UPPER(province), 'UNKNOWN')
            || '|'
            || COALESCE(UPPER(authority_district), 'UNKNOWN')
        ) AS geography_key,

        CASE
            WHEN okas_code_clean IS NOT NULL
                THEN 'PRODUCT:' || okas_code_clean
            ELSE 'PRODUCT:UNKNOWN'
        END AS product_key,

        CASE
            WHEN method_code IS NOT NULL
                THEN 'METHOD_CODE:' || method_code
            WHEN method IS NOT NULL
                THEN 'METHOD_NAME:' || MD5(UPPER(method))
            ELSE 'METHOD:UNKNOWN'
        END AS procedure_key,

        CASE
            WHEN procurement_type IS NOT NULL
                THEN 'TYPE:' || MD5(UPPER(procurement_type))
            ELSE 'TYPE:UNKNOWN'
        END AS procurement_type_key
    FROM contract_histories AS h
    """
)

print("Created temporary analytical table: final_contracts")

In [ ]:
# Validate histories and manually inspect a few returning relationships.

history_validation = con.execute(
    """
    SELECT
        COUNT(*) AS final_rows,
        COUNT(*) FILTER (
            WHERE previous_supplier_wins_nationally < 0
        ) AS negative_previous_national_wins,
        COUNT(*) FILTER (
            WHERE previous_supplier_buyer_wins < 0
        ) AS negative_previous_buyer_wins,
        COUNT(*) FILTER (
            WHERE buyer_supplier_relationship_days < 0
        ) AS negative_relationship_days,
        COUNT(*) FILTER (
            WHERE supplier_status = 'New'
        ) AS observed_new_supplier_rows,
        COUNT(*) FILTER (
            WHERE supplier_status = 'Returning'
        ) AS observed_returning_supplier_rows
    FROM final_contracts
    """
).df()

history_validation

In [ ]:
# Manual history sample: verify that previous-win counts increase in order.

history_sample = con.execute(
    """
    WITH frequent_relationship AS (
        SELECT supplier_clean, authority_id
        FROM final_contracts
        WHERE supplier_clean IS NOT NULL
          AND authority_id IS NOT NULL
          AND analysis_date IS NOT NULL
        GROUP BY supplier_clean, authority_id
        HAVING COUNT(*) >= 5
        ORDER BY COUNT(*) DESC
        LIMIT 1
    )
    SELECT
        f.analysis_date,
        f.contract_id,
        f.ikn,
        f.supplier_clean,
        f.authority_id,
        f.previous_supplier_wins_nationally,
        f.previous_supplier_buyer_wins,
        f.buyer_supplier_relationship_days
    FROM final_contracts AS f
    INNER JOIN frequent_relationship AS r
        ON f.supplier_clean = r.supplier_clean
       AND f.authority_id = r.authority_id
    ORDER BY
        f.analysis_date,
        f.contract_id,
        f.tender_id,
        f.detail_tender_id
    LIMIT 30
    """
).df()

history_sample

## Phase F findings

- `final_contracts` contains **2,370,736 rows**, so the history construction preserves the source grain.
- There are **230,245 first-observed supplier rows** and **2,065,352 returning-supplier rows**.
- The remaining **75,139 rows** have no supplier and therefore receive no new/returning classification.
- No negative previous-win counts or negative relationship durations were produced.
- The manual history sample confirms that previous-win counts increase under the documented deterministic ordering.

These are observed-history measures within this dataset, not proof that a supplier had no procurement activity before the dataset begins.

# Phase G — Descriptive market competition and HHI

The descriptive market is:

`analysis year × province × two-digit OKAS group`

This grouping supports exploration; it is **not** claimed to be a legal or antitrust market definition.

Market shares and HHI use nonnegative contract value among records with a known supplier. HHI is provided on both the `0–1` and `0–10,000` scales.

In [ ]:
# Create one row per market and supplier.

con.execute(
    """
    CREATE OR REPLACE VIEW market_supplier_value AS
    SELECT
        analysis_year,
        province,
        okas2,
        supplier_key,
        SUM(contract_price) AS supplier_contract_value,
        COUNT(*) AS supplier_contract_rows
    FROM final_contracts
    WHERE analysis_year IS NOT NULL
      AND province IS NOT NULL
      AND okas2 IS NOT NULL
      AND supplier_clean IS NOT NULL
      AND contract_price >= 0
    GROUP BY
        analysis_year,
        province,
        okas2,
        supplier_key
    """
)

print("Created DuckDB view: market_supplier_value")

In [ ]:
# Create supplier shares within each descriptive market.

con.execute(
    """
    CREATE OR REPLACE VIEW market_supplier_shares AS
    SELECT
        *,
        supplier_contract_value
            / NULLIF(
                SUM(supplier_contract_value) OVER (
                    PARTITION BY analysis_year, province, okas2
                ),
                0
            ) AS supplier_value_share
    FROM market_supplier_value
    """
)

print("Created DuckDB view: market_supplier_shares")

In [ ]:
# Aggregate market structure, competition, and rebate measures.

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE agg_market_year AS
    WITH concentration AS (
        SELECT
            analysis_year,
            province,
            okas2,
            COUNT(*) AS supplier_count,
            SUM(supplier_contract_value) AS known_supplier_contract_value,
            MAX(supplier_value_share) AS top_supplier_share,
            SUM(POWER(supplier_value_share, 2)) AS HHI_0_1
        FROM market_supplier_shares
        GROUP BY analysis_year, province, okas2
    ),
    competition AS (
        SELECT
            analysis_year,
            province,
            okas2,
            COUNT(*) AS contract_rows,
            SUM(contract_price) FILTER (
                WHERE contract_price >= 0
            ) AS total_contract_value,
            AVG(CAST(single_bid AS INTEGER)) FILTER (
                WHERE NOT valid_offers_missing
            ) AS single_bid_share,
            AVG(num_valid_offers) AS mean_valid_bids,
            AVG(rebate) AS mean_rebate,
            MEDIAN(rebate) AS median_rebate
        FROM final_contracts
        WHERE analysis_year IS NOT NULL
          AND province IS NOT NULL
          AND okas2 IS NOT NULL
        GROUP BY analysis_year, province, okas2
    )
    SELECT
        c.analysis_year,
        c.province,
        c.okas2,
        c.contract_rows,
        c.total_contract_value,
        concentration.supplier_count,
        concentration.known_supplier_contract_value,
        concentration.top_supplier_share,
        concentration.HHI_0_1,
        concentration.HHI_0_1 * 10000 AS HHI_0_10000,
        c.single_bid_share,
        c.mean_valid_bids,
        c.mean_rebate,
        c.median_rebate
    FROM competition AS c
    LEFT JOIN concentration
        USING (analysis_year, province, okas2)
    """
)

print("Created temporary aggregate table: agg_market_year")

In [ ]:
# Validate HHI bounds and inspect the most concentrated markets.

hhi_validation = con.execute(
    """
    SELECT
        COUNT(*) AS markets,
        COUNT(*) FILTER (
            WHERE HHI_0_1 < 0 OR HHI_0_1 > 1.0000001
        ) AS HHI_outside_zero_one,
        MIN(HHI_0_1) AS minimum_HHI,
        MAX(HHI_0_1) AS maximum_HHI,
        COUNT(*) FILTER (
            WHERE supplier_count = 1
        ) AS one_supplier_markets
    FROM agg_market_year
    """
).df()

display(hhi_validation)

most_concentrated_markets = con.execute(
    """
    SELECT *
    FROM agg_market_year
    WHERE supplier_count >= 2
    ORDER BY HHI_0_1 DESC, total_contract_value DESC
    LIMIT 20
    """
).df()

most_concentrated_markets

## Phase G findings

- The descriptive definition `analysis year × province × OKAS2` produces **46,634 markets**.
- HHI ranges from **0.004551 to 1.000000**, with **zero values outside the valid 0–1 interval**.
- **5,365 markets** contain only one observed supplier.
- HHI and supplier shares use nonnegative contract value among records with known supplier identity.
- The grouping is designed for exploratory comparison and is not asserted to be a legal or antitrust market definition.

The market table also includes supplier count, top-supplier share, single-bid share, mean valid bids, mean rebate, and median rebate.

# Phase H — Build and export the Power BI star schema

The fact table keeps procurement observations and measures. Dimension tables hold reusable descriptive attributes.

The generated Parquet files are local analytical outputs. They are ignored by Git and must not be committed before disclosure review.

In [ ]:
# Build the fact view with foreign keys for the dimensions.

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE fact_contracts AS
    SELECT
        ROW_NUMBER() OVER () AS contract_row_key,

        tender_id,
        detail_tender_id,
        contract_id,
        announcement_id,
        ikn,
        authority_key,
        supplier_key,
        geography_key,
        product_key,
        procedure_key,
        procurement_type_key,

        analysis_date,
        analysis_date_source,
        contract_date_clean,
        tender_date,
        tender_announcement_date,
        result_announcement_date,
        tender_datetime,
        analysis_year,
        analysis_month,
        contract_year,
        contract_month,

        contract_price,
        total_estimated_cost,
        lot_estimated_cost,
        analytical_estimated_cost,
        price_to_estimate_ratio,
        rebate,
        num_offers,
        num_valid_offers,
        total_lots_in_tender,
        okas_count,
        document_count,
        days_announce_to_contract,

        is_electronic,
        is_partial,
        is_invitation_only,
        lot_estimate_missing,
        is_multi_lot,
        valid_offers_missing,
        zero_valid_offers,
        single_bid,
        positive_valid_offers,
        contract_above_estimate,
        extreme_price_to_estimate_ratio,
        tender_date_outside_expected_coverage,
        contract_date_outside_expected_coverage,
        analysis_date_outside_expected_coverage,

        previous_supplier_wins_nationally,
        previous_supplier_buyer_wins,
        previous_supplier_okas2_wins,
        previous_supplier_buyer_okas2_wins,
        first_observed_supplier_year,
        supplier_status,
        first_observed_buyer_supplier_date,
        buyer_supplier_relationship_days
    FROM final_contracts
    """
)

print("Created temporary fact table: fact_contracts")

In [ ]:
# Build dimensions. ROW_NUMBER selects one representative description per key.

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_authority AS
    SELECT
        authority_key,
        authority_id,
        authority,
        parent_authority,
        top_authority_code,
        top_authority_name
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY authority_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_supplier AS
    SELECT
        supplier_key,
        supplier_clean AS supplier_name,
        first_observed_supplier_year
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY supplier_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_geography AS
    SELECT
        geography_key,
        province,
        authority_district
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY geography_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_product AS
    SELECT
        product_key,
        okas_code_raw,
        okas_code_clean,
        okas2,
        okas_desc
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY product_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_procedure AS
    SELECT
        procedure_key,
        method_code,
        method,
        scope
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY procedure_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_procurement_type AS
    SELECT
        procurement_type_key,
        procurement_type
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY procurement_type_key
                ORDER BY analysis_date DESC NULLS LAST, contract_id DESC
            ) AS dimension_row_number
        FROM final_contracts
    )
    WHERE dimension_row_number = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_date AS
    WITH date_bounds AS (
        SELECT
            MIN(analysis_date) AS minimum_date,
            MAX(analysis_date) AS maximum_date
        FROM final_contracts
        WHERE analysis_date IS NOT NULL
    ),
    calendar AS (
        SELECT generated_date::DATE AS date_day
        FROM date_bounds,
        GENERATE_SERIES(
            minimum_date,
            maximum_date,
            INTERVAL 1 DAY
        ) AS dates(generated_date)
    )
    SELECT
        date_day AS date,
        EXTRACT(YEAR FROM date_day)::INTEGER AS year,
        EXTRACT(QUARTER FROM date_day)::INTEGER AS quarter_number,
        'Q' || EXTRACT(QUARTER FROM date_day)::INTEGER AS quarter,
        EXTRACT(MONTH FROM date_day)::INTEGER AS month_number,
        STRFTIME(date_day, '%B') AS month_name,
        STRFTIME(date_day, '%Y-%m') AS year_month,
        EXTRACT(DAY FROM date_day)::INTEGER AS day_of_month,
        EXTRACT(ISODOW FROM date_day)::INTEGER AS iso_day_of_week,
        STRFTIME(date_day, '%A') AS day_name
    FROM calendar
    """
)

print("Created all temporary dimension tables.")

## Geography validation and map coordinates

The raw province–district combinations are not assumed to be correct. This section:

- downloads reproducible 2021 Türkiye ADM1 and ADM2 boundary references when they are absent;
- converts literal `None`, `NULL`, and `NaN` names to missing values for matching;
- standardizes Turkish characters and central-district aliases;
- validates province–district combinations;
- infers or corrects a province only when a district name is globally unique;
- falls back to a province representative point when the district cannot be matched;
- calculates representative points for map display;
- preserves the raw province and district values for auditing;
- adds match-status and review flags.

Reference: geoBoundaries Türkiye 2021, ADM1 and ADM2. The source is OSM-based. ADM1 is licensed CC BY-SA 2.0 and ADM2 is licensed ODbL 1.0. Attribution must remain in the portfolio documentation.

Coordinates are representative points inside administrative polygons, not contracting-authority street locations.

In [ ]:
# Download the open geography references only when they are absent.

from urllib.request import urlretrieve

GEOGRAPHY_REFERENCE_DIR = (
    RAW_CSV.parent / "geography_reference"
)
GEOGRAPHY_REFERENCE_DIR.mkdir(parents=True, exist_ok=True)

ADM1_FILE = (
    GEOGRAPHY_REFERENCE_DIR
    / "turkiye_adm1_geoboundaries_2021.geojson"
)
ADM2_FILE = (
    GEOGRAPHY_REFERENCE_DIR
    / "turkiye_adm2_geoboundaries_2021.geojson"
)

geography_downloads = {
    ADM1_FILE: (
        "https://github.com/wmgeolab/geoBoundaries/raw/"
        "9469f09/releaseData/gbOpen/TUR/ADM1/"
        "geoBoundaries-TUR-ADM1.geojson"
    ),
    ADM2_FILE: (
        "https://github.com/wmgeolab/geoBoundaries/raw/"
        "9469f09/releaseData/gbOpen/TUR/ADM2/"
        "geoBoundaries-TUR-ADM2.geojson"
    ),
}

for output_file, download_url in geography_downloads.items():
    if not output_file.exists():
        print("Downloading:", output_file.name)
        urlretrieve(download_url, output_file)
    else:
        print("Reference already exists:", output_file.name)

print("ADM1 reference:", ADM1_FILE)
print("ADM2 reference:", ADM2_FILE)

In [ ]:
# Prepare province and district reference points.

import re
import unicodedata

import geopandas as gpd


def normalize_turkish_name(value):
    if pd.isna(value):
        return None

    text = str(value).strip()
    if not text or text.upper() in {"NONE", "NULL", "NAN"}:
        return None

    text = text.upper().translate(
        str.maketrans(
            {
                "Ç": "C",
                "Ğ": "G",
                "İ": "I",
                "Ö": "O",
                "Ş": "S",
                "Ü": "U",
            }
        )
    )
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip() or None


def normalize_district_name(value):
    text = normalize_turkish_name(value)
    if text is None:
        return None

    for suffix in (
        " MERKEZ ILCE",
        " MERKEZI",
        " DISTRICT",
        " MERKEZ",
    ):
        if text.endswith(suffix):
            text = text.removesuffix(suffix).strip()

    return text or None


def clean_reference_district_display(value, province_reference):
    normalized = normalize_district_name(value)
    province_normalized = normalize_turkish_name(
        province_reference
    )
    if normalized == province_normalized:
        return province_reference

    text = str(value).strip()
    text = re.sub(
        r"\s*\(Merkez İlçe\)\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\s+District\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\s+merkezi\s*$",
        "",
        text,
        flags=re.IGNORECASE,
    )
    return text.strip()


def district_candidates(
    district_normalized,
    province_normalized,
):
    candidates = []

    if district_normalized:
        candidates.append(district_normalized)
        if district_normalized.endswith(" MERKEZ"):
            candidates.append(
                district_normalized.removesuffix(
                    " MERKEZ"
                ).strip()
            )

    if district_normalized == "MERKEZ" and province_normalized:
        candidates.append(province_normalized)

    return list(
        dict.fromkeys(
            candidate
            for candidate in candidates
            if candidate
        )
    )


adm1 = gpd.read_file(ADM1_FILE).to_crs("EPSG:4326")
adm2 = gpd.read_file(ADM2_FILE).to_crs("EPSG:4326")

adm1_reference = adm1[
    ["shapeName", "shapeISO", "shapeID", "geometry"]
].rename(
    columns={
        "shapeName": "province_reference",
        "shapeISO": "province_iso",
        "shapeID": "province_reference_id",
    }
)
adm1_reference["province_normalized"] = (
    adm1_reference["province_reference"].map(
        normalize_turkish_name
    )
)
adm1_reference["province_point"] = (
    adm1_reference.geometry.representative_point()
)
adm1_reference["province_latitude"] = (
    adm1_reference["province_point"].y
)
adm1_reference["province_longitude"] = (
    adm1_reference["province_point"].x
)

adm2_points = adm2[
    ["shapeName", "shapeID", "geometry"]
].rename(
    columns={
        "shapeName": "district_reference",
        "shapeID": "district_reference_id",
    }
)
adm2_points["geometry"] = (
    adm2_points.geometry.representative_point()
)

district_reference = gpd.sjoin(
    adm2_points,
    adm1_reference[
        [
            "province_reference",
            "province_iso",
            "province_reference_id",
            "province_normalized",
            "geometry",
        ]
    ],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

district_reference["district_normalized"] = (
    district_reference["district_reference"].map(
        normalize_district_name
    )
)
district_reference["district_reference_clean"] = (
    district_reference.apply(
        lambda row: clean_reference_district_display(
            row["district_reference"],
            row["province_reference"],
        ),
        axis=1,
    )
)
district_reference["district_latitude"] = (
    district_reference.geometry.y
)
district_reference["district_longitude"] = (
    district_reference.geometry.x
)

print("Reference provinces:", len(adm1_reference))
print("Reference districts:", len(district_reference))

In [ ]:
# Match each raw geography combination to the reference.

province_lookup = {
    row.province_normalized: row
    for row in adm1_reference.itertuples()
}

pair_lookup = {}
district_lookup = {}
for row in district_reference.itertuples():
    pair_lookup[
        (row.province_normalized, row.district_normalized)
    ] = row
    district_lookup.setdefault(
        row.district_normalized,
        [],
    ).append(row)


def match_geography(row):
    province_raw = row["province"]
    district_raw = row["authority_district"]
    province_normalized = normalize_turkish_name(
        province_raw
    )
    district_normalized = normalize_district_name(
        district_raw
    )

    valid_province = province_lookup.get(
        province_normalized
    )
    candidates = district_candidates(
        district_normalized,
        province_normalized,
    )

    for candidate in candidates:
        reference = pair_lookup.get(
            (province_normalized, candidate)
        )
        if reference is not None:
            return {
                "country": "Türkiye",
                "province_clean": (
                    reference.province_reference
                ),
                "district_clean": (
                    reference.district_reference_clean
                ),
                "latitude": reference.district_latitude,
                "longitude": reference.district_longitude,
                "coordinate_level": (
                    "district_representative_point"
                ),
                "geography_match_status": (
                    "exact_province_district"
                ),
                "province_iso": reference.province_iso,
                "province_reference_id": (
                    reference.province_reference_id
                ),
                "district_reference_id": (
                    reference.district_reference_id
                ),
            }

    unique_matches = []
    for candidate in candidates:
        matches = district_lookup.get(candidate, [])
        if len(matches) == 1:
            unique_matches.append(matches[0])

    unique_matches = {
        match.district_reference_id: match
        for match in unique_matches
    }

    if len(unique_matches) == 1:
        reference = next(iter(unique_matches.values()))
        if valid_province is None:
            status = (
                "province_inferred_from_unique_district"
            )
        elif (
            reference.province_normalized
            != province_normalized
        ):
            status = (
                "province_corrected_from_unique_district"
            )
        else:
            status = "district_alias_matched"

        return {
            "country": "Türkiye",
            "province_clean": reference.province_reference,
            "district_clean": (
                reference.district_reference_clean
            ),
            "latitude": reference.district_latitude,
            "longitude": reference.district_longitude,
            "coordinate_level": (
                "district_representative_point"
            ),
            "geography_match_status": status,
            "province_iso": reference.province_iso,
            "province_reference_id": (
                reference.province_reference_id
            ),
            "district_reference_id": (
                reference.district_reference_id
            ),
        }

    if valid_province is not None:
        status = (
            "province_only"
            if district_normalized is None
            else "district_unmatched_province_fallback"
        )
        return {
            "country": "Türkiye",
            "province_clean": (
                valid_province.province_reference
            ),
            "district_clean": None,
            "latitude": valid_province.province_latitude,
            "longitude": valid_province.province_longitude,
            "coordinate_level": (
                "province_representative_point"
            ),
            "geography_match_status": status,
            "province_iso": valid_province.province_iso,
            "province_reference_id": (
                valid_province.province_reference_id
            ),
            "district_reference_id": None,
        }

    return {
        "country": "Türkiye",
        "province_clean": None,
        "district_clean": None,
        "latitude": None,
        "longitude": None,
        "coordinate_level": None,
        "geography_match_status": "unmatched",
        "province_iso": None,
        "province_reference_id": None,
        "district_reference_id": None,
    }


base_geography = con.execute(
    """
    SELECT
        geography_key,
        province,
        authority_district
    FROM dim_geography
    """
).df()

geography_matches = pd.DataFrame(
    [
        match_geography(row)
        for _, row in base_geography.iterrows()
    ]
)

enriched_geography = pd.concat(
    [
        base_geography.rename(
            columns={
                "province": "province_raw",
                "authority_district": (
                    "authority_district_raw"
                ),
            }
        ).reset_index(drop=True),
        geography_matches.reset_index(drop=True),
    ],
    axis=1,
)

enriched_geography["geography_reference"] = (
    "geoBoundaries TUR ADM1/ADM2 2021; "
    "OSM-based"
)
enriched_geography["map_eligible"] = (
    enriched_geography["latitude"].notna()
    & enriched_geography["longitude"].notna()
)
enriched_geography["geography_review_required"] = (
    ~enriched_geography[
        "geography_match_status"
    ].isin(
        [
            "exact_province_district",
            "district_alias_matched",
            "province_only",
        ]
    )
)

con.register(
    "enriched_geography_dataframe",
    enriched_geography,
)
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE dim_geography AS
    SELECT *
    FROM enriched_geography_dataframe
    """
)
con.unregister("enriched_geography_dataframe")

print("Created enriched temporary table: dim_geography")

In [ ]:
# Validate map coverage, coordinates, and fact relationships.

geography_match_summary = con.execute(
    """
    SELECT
        geography_match_status,
        COUNT(*) AS geography_rows,
        COUNT(*) FILTER (
            WHERE map_eligible
        ) AS map_eligible_rows,
        COUNT(*) FILTER (
            WHERE geography_review_required
        ) AS review_required_rows
    FROM dim_geography
    GROUP BY geography_match_status
    ORDER BY geography_rows DESC
    """
).df()

display(geography_match_summary)

geography_validation = con.execute(
    """
    SELECT
        COUNT(*) AS geography_rows,
        COUNT(DISTINCT geography_key) AS distinct_keys,
        COUNT(*) FILTER (
            WHERE map_eligible
        ) AS rows_with_coordinates,
        COUNT(*) FILTER (
            WHERE NOT map_eligible
        ) AS rows_without_coordinates,
        COUNT(*) FILTER (
            WHERE latitude IS NOT NULL
              AND NOT latitude BETWEEN 35 AND 43
        ) AS latitude_outside_turkiye_range,
        COUNT(*) FILTER (
            WHERE longitude IS NOT NULL
              AND NOT longitude BETWEEN 25 AND 45
        ) AS longitude_outside_turkiye_range
    FROM dim_geography
    """
).df()

geography_validation

In [ ]:
# Validate fact rows, totals, year range, and dimension-key uniqueness.

star_schema_summary = con.execute(
    """
    SELECT
        COUNT(*) AS fact_rows,
        COUNT(DISTINCT contract_row_key) AS distinct_fact_row_keys,
        COUNT(DISTINCT contract_id) AS distinct_contract_ids,
        SUM(contract_price) AS total_contract_value,
        MIN(analysis_year) AS minimum_analysis_year,
        MAX(analysis_year) AS maximum_analysis_year
    FROM fact_contracts
    """
).df()

display(star_schema_summary)

dimension_key_check = con.execute(
    """
    SELECT 'dim_authority' AS table_name,
           COUNT(*) AS rows,
           COUNT(DISTINCT authority_key) AS distinct_keys
    FROM dim_authority
    UNION ALL
    SELECT 'dim_supplier', COUNT(*), COUNT(DISTINCT supplier_key)
    FROM dim_supplier
    UNION ALL
    SELECT 'dim_geography', COUNT(*), COUNT(DISTINCT geography_key)
    FROM dim_geography
    UNION ALL
    SELECT 'dim_product', COUNT(*), COUNT(DISTINCT product_key)
    FROM dim_product
    UNION ALL
    SELECT 'dim_procedure', COUNT(*), COUNT(DISTINCT procedure_key)
    FROM dim_procedure
    UNION ALL
    SELECT 'dim_procurement_type',
           COUNT(*),
           COUNT(DISTINCT procurement_type_key)
    FROM dim_procurement_type
    UNION ALL
    SELECT 'dim_date', COUNT(*), COUNT(DISTINCT date)
    FROM dim_date
    """
).df()

dimension_key_check

In [ ]:
# Validate that every fact foreign key finds a dimension row.

unmatched_foreign_keys = con.execute(
    """
    SELECT
        COUNT(*) FILTER (
            WHERE a.authority_key IS NULL
        ) AS unmatched_authority_keys,
        COUNT(*) FILTER (
            WHERE s.supplier_key IS NULL
        ) AS unmatched_supplier_keys,
        COUNT(*) FILTER (
            WHERE g.geography_key IS NULL
        ) AS unmatched_geography_keys,
        COUNT(*) FILTER (
            WHERE p.product_key IS NULL
        ) AS unmatched_product_keys,
        COUNT(*) FILTER (
            WHERE pr.procedure_key IS NULL
        ) AS unmatched_procedure_keys,
        COUNT(*) FILTER (
            WHERE pt.procurement_type_key IS NULL
        ) AS unmatched_procurement_type_keys,
        COUNT(*) FILTER (
            WHERE f.analysis_date IS NOT NULL
              AND d.date IS NULL
        ) AS unmatched_date_keys
    FROM fact_contracts AS f
    LEFT JOIN dim_authority AS a
        ON f.authority_key = a.authority_key
    LEFT JOIN dim_supplier AS s
        ON f.supplier_key = s.supplier_key
    LEFT JOIN dim_geography AS g
        ON f.geography_key = g.geography_key
    LEFT JOIN dim_product AS p
        ON f.product_key = p.product_key
    LEFT JOIN dim_procedure AS pr
        ON f.procedure_key = pr.procedure_key
    LEFT JOIN dim_procurement_type AS pt
        ON f.procurement_type_key = pt.procurement_type_key
    LEFT JOIN dim_date AS d
        ON f.analysis_date = d.date
    """
).df()

unmatched_foreign_keys

In [ ]:
# Stop before export if the structural star-schema checks fail.

fact_rows = int(star_schema_summary.loc[0, "fact_rows"])
distinct_fact_keys = int(
    star_schema_summary.loc[0, "distinct_fact_row_keys"]
)

assert fact_rows == raw_count, "Fact row count does not match the raw data."
assert fact_rows == distinct_fact_keys, "Fact row keys are not unique."
assert (
    dimension_key_check["rows"]
    == dimension_key_check["distinct_keys"]
).all(), "At least one dimension key is duplicated."
assert (
    unmatched_foreign_keys.fillna(0).iloc[0] == 0
).all(), "At least one fact foreign key is unmatched."

print("All structural validation checks passed.")

In [ ]:
# Export only the tables Power BI needs.
# Existing files with the same names are replaced.

exports = {
    "fact_contracts": "fact_contracts.parquet",
    "dim_date": "dim_date.parquet",
    "dim_authority": "dim_authority.parquet",
    "dim_supplier": "dim_supplier.parquet",
    "dim_geography": "dim_geography.parquet",
    "dim_product": "dim_product.parquet",
    "dim_procedure": "dim_procedure.parquet",
    "dim_procurement_type": "dim_procurement_type.parquet",
    "agg_market_year": "agg_market_year.parquet",
}

for view_name, file_name in exports.items():
    output_path = (PROCESSED_DIR / file_name).as_posix()
    output_path_sql = output_path.replace("'", "''")

    con.execute(
        f"""
        COPY (
            SELECT *
            FROM {view_name}
        )
        TO '{output_path_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(f"Exported {view_name}: {output_path}")

In [ ]:
# Confirm that every expected Parquet file exists and record its size.

export_file_check = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "file_name": file_name,
            "exists": (PROCESSED_DIR / file_name).exists(),
            "size_mb": round(
                (PROCESSED_DIR / file_name).stat().st_size
                / (1024 ** 2),
                2,
            )
            if (PROCESSED_DIR / file_name).exists()
            else None,
        }
        for table_name, file_name in exports.items()
    ]
)

export_file_check

In [ ]:
# Read the exported files back through DuckDB for a final independent check.

export_validation_parts = []

for table_name, file_name in exports.items():
    parquet_path = (PROCESSED_DIR / file_name).as_posix()
    parquet_path_sql = parquet_path.replace("'", "''")

    export_validation_parts.append(
        f"""
        SELECT
            '{table_name}' AS table_name,
            COUNT(*) AS exported_rows
        FROM read_parquet('{parquet_path_sql}')
        """
    )

exported_row_counts = con.execute(
    "\nUNION ALL\n".join(export_validation_parts)
).df()

exported_row_counts

## Phase H findings

- Fact rows: **2,370,736**.
- Distinct fact-row keys: **2,370,736**.
- Distinct source contract IDs: **2,370,077**.
- Total nominal contract value: **7,647,302,084,315.01** in the source currency units.
- Observed analysis-date range: **2010-10-13 to 2026-04-01**.
- Dimension keys are unique in every dimension.
- Unmatched fact foreign keys: **0** for authority, supplier, geography, product, procedure, procurement type, and date.
- The enriched geography dimension contains **1,376 unique keys**.
- **1,358 geography keys** have representative-point coordinates and **18** remain unplottable.
- Geography matching results: **969 exact province–district matches**, **165 provinces inferred from a unique district**, **110 provinces corrected from a unique district**, **81 province-only rows**, **33 district-unmatched province fallbacks**, and **18 unmatched rows**.
- All fact geography keys remain matched after enrichment.
- Coordinates are administrative representative points, not authority addresses.
- Exported tables and rows:
  - `fact_contracts`: **2,370,736**
  - `dim_date`: **5,650**
  - `dim_authority`: **33,253**
  - `dim_supplier`: **230,246**
  - `dim_geography`: **1,376**
  - `dim_product`: **6,930**
  - `dim_procedure`: **28**
  - `dim_procurement_type`: **4**
  - `agg_market_year`: **46,634**
- All nine Parquet files were successfully written and read back through DuckDB.
- The fact Parquet is approximately **308 MB** before adding the final quality flags; generated file sizes may vary slightly after reruns.

All structural assertions pass. The Python/DuckDB data model is ready for Power BI, subject to the documented date, rebate, geographic-category, and disclosure caveats.

# Task 13 — Git workflow

Git commands are intentionally **not executed inside this notebook**. Run them in the VS Code PowerShell terminal after saving the notebook.

The generated files under `data/processed/`, the raw data, `.env`, Excel audit files, and Power BI files containing data must remain untracked.

## Step 1 — Clear private outputs and save

Notebook outputs can embed supplier names and sample source rows. In VS Code, select **Clear All Outputs**, then save with `Ctrl+S`. The completed aggregate findings remain recorded in Markdown.

## Step 2 — Inspect repository status

Run:

```powershell
git status --short
```

Expected:

- `notebooks/procurement_project.ipynb` may be modified or untracked.
- generated files inside `data/processed` should not appear.
- `.env`, the raw CSV, and Excel audit files should not appear.

## Step 3 — Verify sensitive/generated files are ignored

Run:

```powershell
git check-ignore -v .env
git check-ignore -v data/raw/merged_contract_level_2010_2024.csv
git check-ignore -v data/processed/fact_contracts.parquet
git check-ignore -v excel/procurement_data_audit.xlsx
```

Each command should print the matching `.gitignore` rule.

## Step 4 — Review exactly what will be committed

If the notebook is already tracked:

```powershell
git diff -- notebooks/procurement_project.ipynb
```

The final notebook should contain code and aggregate Markdown findings but no embedded code-cell outputs. Confirm that no raw data or private local path appears in the diff.

## Step 5 — Stage only the notebook

```powershell
git add notebooks/procurement_project.ipynb
git status
```

Under **Changes to be committed**, only the intended project files should appear.

## Step 6 — Commit

```powershell
git commit -m "Build procurement analytics and Power BI data model"
```

## Step 7 — Synchronize and push

```powershell
git pull --rebase origin main
git push origin main
```

If `git pull --rebase` reports a conflict, stop and resolve it before pushing. Do not use destructive reset commands.

## Step 8 — Confirm

```powershell
git status
```

Expected final message:

```text
Your branch is up to date with 'origin/main'.
nothing to commit, working tree clean
```

After this confirmation, the Python/DuckDB stage is complete and the next project stage is Power BI.